In [1]:
# Ô 1 — gói, Java, mốc giờ
import subprocess, sys, os, time
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y openjdk-8-jre-headless >/dev/null 2>&1; java -version")
sh(f"{sys.executable} -m pip install -q 'trl==0.29.1' 'transformers==5.18.0' 'peft==0.21.1' accelerate bitsandbytes pycocoevalcap sacrebleu 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y -q torchao")
import importlib.util; assert importlib.util.find_spec("torchao") is None, "DỪNG: torchao còn"
sh(f'{sys.executable} -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"')


openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.2 MB/s eta 0:00:00



5.18.0 0.21.1 2.11.0+cu128 2 Tesla T4



0

In [2]:
# Ô 2 — đường dẫn, md5, hàm chạy song song
import glob, json, shutil, hashlib, re
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
MD5 = {
 "build_branch_data.py": "619e63e123a6dbf60086e65ee94a3912",
 "ex_test_k4.jsonl": "4d8d1646402aeda72f3a826c3d663702",
 "ex_val_k4.jsonl": "3a7ae1e2360659c684234e9d6fbcb866",
 "grpo_spice.py": "07ea87b6d156d1faa391a4274dffd7cf",
 "pred_ck500_test.jsonl": "328847ac96fa3104f76cd997f4091bcd",
 "pred_ck500_vallon.jsonl": "d092faf6f862eec5db71110e6b5963e2",
 "ra_build.py": "7e4ee74274037b83d11db5d63c5e66a8",
 "ra_exemplars.py": "319857d87a6d5fdf8578886ffbb1ac0f",
 "ra_gen.py": "d3969f6105b53b9eae81e7ce521cbb27",
 "ra_score.py": "04aab5e14ee01468b2c6fb8683bc2f8a",
 "ra_train.py": "217c75f188046c4bd279bb11fb1757e4",
 "train_tru_val.jsonl": "edbb231a58574acf6e1968a44d1fa6c6"
}
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ra_train.py", recursive=True)
       if os.path.exists(os.path.join(os.path.dirname(p), "train_tru_val.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một dataset ra-sft-script, thấy {SRC}"
SRC = SRC[0]
for f, h in MD5.items():
    assert md5(f"{SRC}/{f}") == h, f"DỪNG: {f} lệch md5"
    if f.endswith(".py"):
        shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "p1_train_rows.jsonl" in f)
CK = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
            if md5(p) == "491fa6677340393f1e4464c08a0cec98")
CK = [c for c in CK if not c.strip("/").endswith("ref")]
assert len(CK) == 1, f"DỪNG: không thấy đúng một adapter ck500, thấy {CK}"
CK500 = CK[0]
MERGED = f"{W}/s1_merged"
def chay(jobs, nhip=120):
    ps = []
    for i, (ten, cmd, gpu) in enumerate(jobs):
        if i: time.sleep(90)
        f = open(f"{W}/{ten}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
        ps.append((ten, subprocess.Popen(cmd, cwd=W, stdout=f, stderr=subprocess.STDOUT, env=env), f))
    while any(p.poll() is None for _, p, _ in ps):
        time.sleep(nhip)
        for ten, p, _ in ps:
            L = [l.strip() for l in open(f"{W}/{ten}.log", errors="ignore") if l.strip()]
            print(f" {(time.time() - T_NB) / 3600:5.2f} h · {ten} · {'xong' if p.poll() is not None else 'chạy'} · {L[-1][:110] if L else '..'}", flush=True)
    for ten, p, f in ps:
        f.close()
        print(f"--- {ten} · mã thoát {p.returncode} ---\n" + "".join(open(f"{W}/{ten}.log", errors="ignore").readlines()[-10:]), flush=True)
    return {ten: p.returncode for ten, p, _ in ps}
def hoa():
    r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); assert r.returncode == 0, "DỪNG: hoà S1 lỗi"
print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500, flush=True)


SRC    /kaggle/input/datasets/trangphngngc/ra-sft-script
BUNDLE /kaggle/input/datasets/trangphngngc/fgrb-p1-bundle/fgrb_p1_bundle
CK500  /kaggle/input/datasets/trangphngngc/grpo-spice-ck500


In [3]:
# Ô hoà S1
hoa()
assert os.path.isdir(MERGED)

[hoà] xong → /kaggle/working/s1_merged · dtype torch.float16



In [4]:
# Ô C1-1: chép mã, hàm chạy nền
import os, json, time, shutil, subprocess, glob
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"

def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g:
            return g[0]
    raise FileNotFoundError(duoi)

CW = os.path.dirname(tim("cw_gen.py"))
shutil.copytree(SRC, RA, dirs_exist_ok=True)
for f in os.listdir(CW):
    shutil.copy(f"{CW}/{f}", RA)
print("CW", CW, sorted(os.listdir(RA)))
T0 = T_NB  # tính giờ từ đầu notebook (gồm pip + hoà S1) để chắc không chạm trần 12 h

def chay(script, args, log, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"),
                            stderr=subprocess.STDOUT, env=env, cwd=RA, start_new_session=True)

def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120)
        print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h",
              [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]

CW /kaggle/input/datasets/trangphngngc/cw-ra-309 ['build_branch_data.py', 'cw_build.py', 'cw_chon_tau.py', 'cw_gen.py', 'cw_ghep.py', 'ex_test_k4.jsonl', 'ex_val_k4.jsonl', 'grpo_spice.py', 'pred_ck500_test.jsonl', 'pred_ck500_vallon.jsonl', 'ra_build.py', 'ra_exemplars.py', 'ra_gen.py', 'ra_score.py', 'ra_train.py', 'train_tru_val.jsonl']


In [5]:
# Ô C1-2: kiểm đường sinh — 20 bước val phải trùng câu ck500 đã có (~3 phút)
p = chay("cw_gen.py", ["--mode", "tu", "--merged", M, "--ckpt", CK500, "--recs", f"{BUNDLE}/p1_val_rows.jsonl",
                         "--ocr", f"{BUNDLE}/ocr.jsonl", "--images", f"{BUNDLE}/images", "--chi-cham", "--n", "20",
                         "--out", f"{W}/kiem20.jsonl"], f"{W}/kiem20.log", 0)
doi([p], [f"{W}/kiem20.log"])
A = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{W}/kiem20.jsonl"))}
B = {(d["episode_id"], d["step_id"]): d["pred"] for d in map(json.loads, open(f"{SRC}/pred_ck500_vallon.jsonl"))}
trung = sum(A[k] == B[k] for k in A)
print(f"[kiểm ck500] {trung}/{len(A)}")
assert trung >= 18, "⛔ đường sinh lệch ck500 — dừng, gửi log cho chat"

16:01 0.11 h [['[xong] 20 dòng trong /kaggle/working/kiem20.jsonl']]


[kiểm ck500] 20/20


In [6]:
# Ô C1-3: câu greedy ck500 trên hàng dạy, câu nhắc KHÔNG khối, hai GPU (~1,6 h)
logs = [f"{W}/tu_{i}.log" for i in (0, 1)]
ps = [chay("cw_gen.py", ["--mode", "tu", "--merged", M, "--ckpt", CK500, "--recs", f"{BUNDLE}/p1_train_rows.jsonl",
                         "--ocr", f"{BUNDLE}/ocr.jsonl", "--images", f"{BUNDLE}/images", "--out", f"{W}/tu_{i}.jsonl",
                         "--shard", f"{i}/2"], logs[i], i) for i in (0, 1)]
doi(ps, logs)
with open(f"{W}/pred_ck500_train.jsonl", "w") as f:
    for i in (0, 1):
        f.write(open(f"{W}/tu_{i}.jsonl").read())
n = sum(1 for _ in open(f"{W}/pred_ck500_train.jsonl"))
print("câu ck500 trên hàng dạy:", n)
assert n >= 3900, "⛔ thiếu câu — xem tu_0.log / tu_1.log"

16:03 0.14 h [['  warnings.warn('], ['  warnings.warn(']]


16:05 0.18 h [['  50/1996 · 2.1 phút · 2.48 s/bước'], ['  50/1995 · 1.9 phút · 2.30 s/bước']]


16:07 0.21 h [['  100/1996 · 4.1 phút · 2.48 s/bước'], ['  100/1995 · 3.9 phút · 2.34 s/bước']]


16:09 0.24 h [['  150/1996 · 6.3 phút · 2.51 s/bước'], ['  150/1995 · 6.0 phút · 2.42 s/bước']]


16:11 0.28 h [['  200/1996 · 8.5 phút · 2.55 s/bước'], ['  200/1995 · 8.2 phút · 2.47 s/bước']]


16:13 0.31 h [['  250/1996 · 10.7 phút · 2.58 s/bước'], ['  250/1995 · 10.4 phút · 2.50 s/bước']]


... (37 outputs left out)

17:29 1.58 h [['  1900/1996 · 85.5 phút · 2.70 s/bước'], ['  1950/1995 · 87.4 phút · 2.69 s/bước']]


17:31 1.61 h [['  1950/1996 · 87.7 phút · 2.70 s/bước'], ['[xong] 1995 dòng trong /kaggle/working/tu_1.jsonl']]


17:33 1.64 h [['[xong] 1996 dòng trong /kaggle/working/tu_0.jsonl'], ['[xong] 1995 dòng trong /kaggle/working/tu_1.jsonl']]


câu ck500 trên hàng dạy: 3991


In [7]:
# Ô C1-4: dựng dữ liệu CW (CPU, ~15 phút)
r = subprocess.run(["python", f"{RA}/cw_build.py", "--bundle", BUNDLE, "--kho", f"{SRC}/train_tru_val.jsonl",
                    "--ck500-train", f"{W}/pred_ck500_train.jsonl", "--out", f"{W}/cw_data"],
                   capture_output=True, text=True, cwd=RA)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0

[xong] {"hang": 3991, "copy": 736, "write": 3255, "copy_pct": 18.4, "ck500_dung_trong_write": 1149, "bo": {"bo_rong": 9}, "kho_buoc": 62880, "phut": 8.7}
 


In [8]:
# Ô C1-5: chạy thử 2 bước, rồi train thật, tự dừng trước 11 h
p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/cw_data/train_cw.jsonl",
                         "--out", f"{W}/thu_cw", "--n", "32"], f"{W}/thu_cw.log", 0)
doi([p], [f"{W}/thu_cw.log"])
print(open(f"{W}/thu_cw.log").read()[-1500:])
con = 11.0 - (time.time()-T0) / 3600 - 0.4
print(f"còn {con:.2f} h cho train")
p = chay("ra_train.py", ["--merged", M, "--ckpt", CK500, "--data", f"{W}/cw_data/train_cw.jsonl",
                         "--out", f"{W}/cw_sft", "--max-hours", f"{con:.2f}"], f"{W}/train_cw.log", 0)
doi([p], [f"{W}/train_cw.log"])
print(open(f"{W}/train_cw.log").read()[-2500:])

17:44 1.82 h [['  bước 1/2 · loss 3.7151 · |g| 6.564 · lr 4.00e-06 · 101.9 s/bước · VRAM 6.7 GiB']]


17:46 1.85 h [['[xong] 2 bước · 3.5 phút → /kaggle/working/thu_cw/final']]


|████▋     | 390/824 [00:01<00:01, 261.64it/s]
Loading weights: 100%|██████████| 824/824 [00:02<00:00, 334.08it/s]
[nạp] /kaggle/working/s1_merged · dtype torch.float16 · 4-bit True · Tesla T4
[adapter] /kaggle/input/datasets/trangphngngc/grpo-spice-ck500 · tham số học 14,966,784 (ck500 14,966,784) · dtype tính torch.float16
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
[kiểm nhãn] mẫu đầu: 1695 token, tính loss 17 token: '[WRITE] Click on the search icon at the bottom right corner of the screen<|im_end|>' · đích '[WRITE] Click on the search icon at the bottom right corner of the screen'
  bước 1/2 · loss 3.7151 · |g| 6.564 · lr 4.00e-06 · 101.9 s/bước · VRAM 6.7 GiB
  bước 2/2 · loss 4.6118 · |g| 7.559 · lr 6.0

17:48 1.89 h [["[kiểm nhãn] mẫu đầu: 1695 token, tính loss 17 token: '[WRITE] Click on the search icon at the bottom right corner of the screen<|im_end|>' · đích '[WRITE] Click on the search icon at the bottom right corner of the screen'"]]


17:50 1.92 h [['  bước 1/250 · loss 3.7151 · |g| 6.564 · lr 4.00e-06 · 108.9 s/bước · VRAM 6.7 GiB']]


17:52 1.95 h [['  bước 1/250 · loss 3.7151 · |g| 6.564 · lr 4.00e-06 · 108.9 s/bước · VRAM 6.7 GiB']]


... (218 outputs left out)

01:10 9.25 h [['  bước 245/250 · loss 0.2284 · |g| 2.118 · lr 2.14e-08 · 107.4 s/bước · VRAM 7.3 GiB']]


01:12 9.29 h [['  bước 245/250 · loss 0.2284 · |g| 2.118 · lr 2.14e-08 · 107.4 s/bước · VRAM 7.3 GiB']]


01:14 9.32 h [['[xong] 250 bước · 447.6 phút → /kaggle/working/cw_sft/final']]


846 · |g| 1.812 · lr 1.26e-05 · 107.4 s/bước · VRAM 7.2 GiB
  bước 115/250 · loss 0.2795 · |g| 1.724 · lr 1.20e-05 · 107.4 s/bước · VRAM 7.2 GiB
  bước 120/250 · loss 0.2614 · |g| 1.667 · lr 1.13e-05 · 107.4 s/bước · VRAM 7.2 GiB
  bước 125/250 · loss 0.2602 · |g| 1.910 · lr 1.07e-05 · 107.3 s/bước · VRAM 7.2 GiB
  bước 130/250 · loss 0.2597 · |g| 1.575 · lr 1.00e-05 · 107.3 s/bước · VRAM 7.2 GiB
  bước 135/250 · loss 0.2198 · |g| 1.119 · lr 9.35e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 140/250 · loss 0.1966 · |g| 2.024 · lr 8.69e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 145/250 · loss 0.2326 · |g| 1.563 · lr 8.05e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 150/250 · loss 0.2320 · |g| 2.253 · lr 7.41e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 155/250 · loss 0.2196 · |g| 1.525 · lr 6.79e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 160/250 · loss 0.2315 · |g| 1.399 · lr 6.17e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bước 165/250 · loss 0.2602 · |g| 1.688 · lr 5.58e-06 · 107.3 s/bước · VRAM 7.2 GiB
  bướ

In [ ]:
# Ô C1-6: xoá thứ nặng rồi đóng gói (zip ghi ra /tmp rồi mới chuyển vào W — tránh lỗi zip tự nén chính nó)
for d in ("thu_cw", "s1_merged", "ra"):
    shutil.rmtree(f"{W}/{d}", ignore_errors=True)
shutil.make_archive("/tmp/ket_qua_309_T", "zip", W, ".")
shutil.move("/tmp/ket_qua_309_T.zip", f"{W}/ket_qua_309_T.zip")
print(os.listdir(W), f"{(time.time()-T0)/3600:.2f} h")
assert os.path.exists(f"{W}/cw_sft/final/adapter_model.safetensors"), "⛔ chưa có cw_sft/final — xem mục 8"